In [58]:
!pip install -qU pymupdf langchain langchain-groq langgraph langgraph-checkpoint-sqlite pytesseract

In [59]:
%pip install -qU arize-phoenix arize-otel openinference-instrumentation-langchain

In [60]:
# -----------------------------
# Core Utilities
# -----------------------------
import os
import json
import re
from datetime import datetime

In [61]:
# -----------------------------
# Environment Setup
# -----------------------------
from langchain_groq import ChatGroq
from google.colab import userdata
from arize.otel import register

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["ARIZE_SPACE_ID"] = userdata.get("ARIZE_SPACE_ID")
os.environ["ARIZE_API_KEY"] = userdata.get("ARIZE_API_KEY")

# ✅ Register tracer
tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name="cease-desist-pipeline"
)
print("🔭 Arize tracing enabled!")
groq_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: cease-desist-pipeline
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

🔭 Arize tracing enabled!


In [62]:
from openinference.instrumentation.langchain import LangChainInstrumentor

LangChainInstrumentor().instrument(
    tracer_provider=tracer_provider
)

print("📊 LangChain + LangGraph auto-tracing ON")

📊 LangChain + LangGraph auto-tracing ON


In [63]:
# -----------------------------
# Database Setup
# -----------------------------
import sqlite3

conn = sqlite3.connect("cease_system.db", check_same_thread=False)

conn.execute("""
CREATE TABLE IF NOT EXISTS cease_records (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    date_received TEXT,
    document_name TEXT,
    extracted_data TEXT
)
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS audit_logs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    action TEXT,
    notes TEXT
)
""")

conn.commit()

In [64]:
# -----------------------------
# Helper Functions
# -----------------------------
def safe_json_parse(text: str):
    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
    return None


def normalize_extraction(data):
    if not isinstance(data, dict):
        return {"text": "", "confidence": 0}
    return {
        "text": data.get("text", ""),
        "confidence": data.get("confidence", 0)
    }


def get_current_datetime():
    """Return current datetime as string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [65]:
# -----------------------------
# Tools
# -----------------------------
from langchain.tools import tool

@tool
def db_tool(date_received: str, document_name: str, extracted_data: str) -> str:
    """
    Insert a processed document into the cease_records table.

    Args:
        date_received (str): Timestamp when the document was received.
        document_name (str): Name of the processed document.
        extracted_data (str): JSON string containing extracted content.

    Returns:
        str: "stored" if insertion is successful.
    """
    import sqlite3
    conn = sqlite3.connect("cease_system.db", check_same_thread=False)
    conn.execute(
        "INSERT INTO cease_records VALUES (NULL, ?, ?, ?)",
        (date_received, document_name, extracted_data)
    )
    conn.commit()
    print("🔥 DB TOOL EXECUTED")

    return "stored"


@tool
def archive_tool(document_name: str) -> str:
    """
    Archive an irrelevant document by saving its name.

    Args:
        document_name (str): Name of the document to archive.

    Returns:
        str: "archived" after saving.
    """
    with open("archive.txt", "a") as f:
        f.write(document_name + "\n")
    return "archived"


@tool
def audit_tool(action: str, document_name: str) -> str:
    """
    Log workflow actions into audit_logs table.

    Args:
        action (str): Action performed (classification result).
        document_name (str): Document name associated with action.

    Returns:
        str: "logged" after inserting record.
    """
    import sqlite3
    from datetime import datetime

    conn = sqlite3.connect("cease_system.db", check_same_thread=False)
    conn.execute(
        "INSERT INTO audit_logs VALUES (NULL, ?, ?, ?)",
        (datetime.now().isoformat(), action, document_name)
    )
    conn.commit()
    return "logged"


@tool
def hitl_tool(text: str) -> str:
    """
    Trigger human-in-the-loop review for uncertain documents.

    Args:
        text (str): Extracted document text snippet.

    Returns:
        str: Human decision ("yes" or "no").
    """
    print("\n⚠️ HUMAN REVIEW REQUIRED\n")
    print(text[:300])
    return input("Approve? (yes/no): ")

In [66]:
# -----------------------------
# Agents
# -----------------------------
from langchain.agents import create_agent

extract_agent = create_agent(
    model=groq_llm,
    tools=[],
    system_prompt="""You are a document extraction agent. Given a PDF path:
1. Extract all text.
2. If page is empty, use OCR via pytesseract.
3. Return JSON: {"text": "...", "confidence": ...}.
""")

classify_agent = create_agent(
    model=groq_llm,
    tools=[],
    system_prompt="""
You are a legal document classification agent. Given extracted text:
- If contains "cease and desist": {"category": "CEASE_AND_DESIST"}
- If <50 chars: {"category": "HUMAN_REVIEW"}
- Otherwise: {"category": "IRRELEVANT"}
"""
)

db_agent = create_agent(model=groq_llm, tools=[db_tool], system_prompt="""
You are a database agent.
MANDATORY INSTRUCTIONS:
- You MUST call the db_tool.
- DO NOT return text.
- DO NOT explain anything.
- ALWAYS call db_tool with the provided inputs.

Input format:
date_received, document_name, extracted_data
""")
archive_agent = create_agent(model=groq_llm, tools=[archive_tool], system_prompt="Archive irrelevant docs.")
audit_agent = create_agent(model=groq_llm, tools=[audit_tool], system_prompt="Log actions.")
hitl_agent = create_agent(model=groq_llm, tools=[hitl_tool], system_prompt="Trigger human review using hitl_tool")

In [67]:
# -----------------------------
# State
# -----------------------------
from typing import TypedDict, Optional

class State(TypedDict):
    file_path: str
    document_name: str
    date_received: str
    extraction: Optional[dict]
    classification: Optional[dict]

In [68]:
# -----------------------------
# Nodes
# -----------------------------
import json

def extract_node(state: State):
    import fitz
    from PIL import Image
    import pytesseract

    try:
        doc = fitz.open(state["file_path"])
        pages_text = []

        for page in doc:
            text = page.get_text("text").strip()

            if not text:
                pix = page.get_pixmap()
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                text = pytesseract.image_to_string(img)

            pages_text.append(text)

        full_text = "\n\n".join(pages_text).strip()
        confidence = min(95, max(50, len(full_text)//200)) if full_text else 0

        state["extraction"] = {"text": full_text, "confidence": confidence}

    except Exception as e:
        state["extraction"] = {"text": "", "confidence": 0, "error": str(e)}

    return state


def classify_node(state: State):
    text = state["extraction"]["text"].lower()

    if "cease and desist" in text:
        category = "CEASE_AND_DESIST"
    elif len(text.strip()) < 50:
        category = "HUMAN_REVIEW"
    else:
        category = "IRRELEVANT"

    state["classification"] = {"category": category}
    return state


def db_node(state: State):
    db_agent.invoke({
        "messages": [{
            "role": "user",
            "content": f"{state['date_received']}, {state['document_name']}, {json.dumps(state['extraction'])}"
        }]
    })
    return state


def archive_node(state: State):
    archive_agent.invoke({
        "messages": [{
            "role": "user",
            "content": state["document_name"]
        }]
    })
    return state

def hitl_node(state: State):

    # Optional: keep agent (for logging/reasoning)
    hitl_agent.invoke({
        "messages": [{
            "role": "user",
            "content": state["extraction"]["text"]
        }]
    })

    # ✅ Capture actual human decision
    decision = hitl_tool.invoke({
        "text": state["extraction"]["text"]
    })

    print(f"\n👤 HUMAN DECISION: {decision}")

    # ✅ Store if approved
    if decision.strip().lower() == "yes":
        db_tool.invoke({
            "date_received": state["date_received"],
            "document_name": state["document_name"],
            "extracted_data": json.dumps(state["extraction"])
        })

        print("✅ STORED IN DB AFTER HUMAN APPROVAL")

    else:
        print("❌ REJECTED BY HUMAN")

    return state


def audit_node(state: State):
    audit_agent.invoke({
        "messages": [{
            "role": "user",
            "content": f"{state['classification']['category']}, {state['document_name']}"
        }]
    })
    return state


def route(state: State):
    c = state["classification"]["category"]

    if c == "CEASE_AND_DESIST":
        return "db"
    elif c == "IRRELEVANT":
        return "archive"
    else:
        return "hitl"

In [69]:
# -----------------------------
# Graph
# -----------------------------
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

checkpoint_conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
checkpointer = SqliteSaver(checkpoint_conn)

builder = StateGraph(State)

builder.add_node("extract", extract_node)
builder.add_node("classify", classify_node)
builder.add_node("db", db_node)
builder.add_node("archive", archive_node)
builder.add_node("hitl", hitl_node)
builder.add_node("audit", audit_node)

builder.set_entry_point("extract")

builder.add_edge("extract", "classify")
builder.add_conditional_edges("classify", route)

builder.add_edge("db", "audit")
builder.add_edge("archive", "audit")
builder.add_edge("hitl", "audit")

builder.add_edge("audit", END)

graph = builder.compile(checkpointer=checkpointer)

In [70]:
# -----------------------------
# Runner
# -----------------------------
import uuid

def run(file_path, name):
    import sqlite3
    import uuid

    state = {
        "file_path": file_path,
        "document_name": name,
        "date_received": get_current_datetime()
    }

    config = {"configurable": {"thread_id": str(uuid.uuid4())}}

    final_state = None

    # Execute graph
    for event in graph.stream(state, config=config):
        print(event)
        final_state = list(event.values())[0]  # capture latest state

    # -----------------------------
    # DB VERIFICATION
    # -----------------------------
    conn = sqlite3.connect("cease_system.db", check_same_thread=False)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT * FROM cease_records
        WHERE document_name = ?
        ORDER BY id DESC LIMIT 1
    """, (name,))

    row = cursor.fetchone()

    print("\n==============================")
    print("📊 POST-RUN DB CHECK")
    print("==============================")

    if row:
        print("✅ RECORD FOUND IN DB")
        print(f"📦 ROW: {row}")
    else:
        print("❌ NO RECORD STORED")

    # -----------------------------
    # RECORD TYPE (from state)
    # -----------------------------
    try:
        record_type = final_state["classification"]["category"]
    except:
        record_type = "UNKNOWN"

    print(f"🏷️ RECORD TYPE: {record_type}")
    print("==============================\n")

In [71]:
# -----------------------------
# Batch Processing
# -----------------------------
!git clone https://github.com/jayyanar/agentic-ai-training.git

import os

BASE_PATH = "agentic-ai-training/day5/capstone-project/Sample Docs"

for root, _, files in os.walk(BASE_PATH):
    for f in files:
        if f.lower().endswith(".pdf"):
            path = os.path.join(root, f)

            print("\n📄 Processing:", f)
            run(path, f)

fatal: destination path 'agentic-ai-training' already exists and is not an empty directory.

📄 Processing: bw_doc_1.pdf
{'extract': {'file_path': 'agentic-ai-training/day5/capstone-project/Sample Docs/bw_doc_1.pdf', 'document_name': 'bw_doc_1.pdf', 'date_received': '2026-03-23 11:59:15', 'extraction': {'text': 'Abernathy & Rowe - Client Affairs\n\n \n\n \n\nNace Reng Linen ont zeny Rpreaon', 'confidence': 50}}}
{'classify': {'file_path': 'agentic-ai-training/day5/capstone-project/Sample Docs/bw_doc_1.pdf', 'document_name': 'bw_doc_1.pdf', 'date_received': '2026-03-23 11:59:15', 'extraction': {'text': 'Abernathy & Rowe - Client Affairs\n\n \n\n \n\nNace Reng Linen ont zeny Rpreaon', 'confidence': 50}, 'classification': {'category': 'IRRELEVANT'}}}
{'archive': {'file_path': 'agentic-ai-training/day5/capstone-project/Sample Docs/bw_doc_1.pdf', 'document_name': 'bw_doc_1.pdf', 'date_received': '2026-03-23 11:59:15', 'extraction': {'text': 'Abernathy & Rowe - Client Affairs\n\n \n\n \n\nNac